RAG评估需要从两个维度同时考量：
- 检索质量 — 搜对了吗？Recall、Precision、MRR
- 生成质量 — 回答对了吗？Faithfulness、Correctness、Relevancy

# RAG评估工具概览

| 工具 | 特点 | 适用场景 |
|:---|:---|:---|
| RAGAS | 开源标准，指标最全 | RAG 系统日常评估 |
| LangSmith | LangChain 官方平台 | 追踪 + 评估一体化 |
| DeepEval | 语法简洁，CI/CD 友好 | 自动化测试流水线 |
| TruLens | 可视化反馈闭环 | 调试和迭代优化 |

其中 $RAGAS$是社区使用最广的RAG评估框架，提供多个核心指标覆盖检索和生成全流程。

## 1.1 评估数据说明

| 字段名 | 含义 | 来源 | 需要 Ground Truth 的指标 |
|:---|:---|:---|:---|
| `user_input` | 用户问题 | 人工或自动生成 | Context Recall / Entities Recall / Noise Sensitivity |
| `retrieved_contexts` | RAG 系统召回的文档列表 | RAG 检索阶段输出 | 所有检索指标 |
| `response` | RAG 系统生成的回答 | RAG 生成阶段输出 | Faithfulness / Answer Relevancy / Noise Sensitivity |
| `reference` | 标准答案 | 人工标注 | Context Recall / Entities Recall / Noise Sensitivity |

$数据集示例$ （ragas.Dataset 单行）：

In [ ]:
{
    "user_input": "孔子的教育思想有哪些？",
    "retrieved_contexts": [
        "孔子主张有教无类，认为人人都应该接受教育...",
        "孔子提出因材施教、启发诱导等教学原则..."
    ],
    "response": "孔子主张有教无类，提出因材施教、启发诱导、学思结合等教学原则。",
    "reference": "孔子主张有教无类，提出因材施教、启发诱导、学思结合、谦虚笃实四大教学原则。教育目的是培养士和君子。"
}

## 1.2 安装RAGAS
pip install ragas

## 1.3 RAGAS 核心指标详解

RAGAS 在"RAG评估"类别下提供8个指标（6个文本 + 2个多模态）。下面逐一说明 6个核心文本指标。

### 1.3.1 Context Precision（上下文精度）
检索返回的文档中，有多少是真正和问题相关的？同时考虑排名——相关文档排得越靠前，分数越高。

公式：

$$\text{Context Precision}@K = \frac{\sum_{i=1}^{K} \bigl( \text{relevant}_i \times \frac{|\{\text{relevant in top }i\}|}{i} \bigr)} {|\{\text{total relevant documents}\}|} $$

举例：
- 用户问："孔子的教育思想有哪些？"
- 检索返回：① 孔子：有教无类、因材施教 ✓ ② 孟子：性善论 ✗ ③ 荀子：性恶论 ✗
- 只有①相关且排在第一位 → Context Precision ≈ 1.0（高）
- 若顺序为：① 孟子 ✗ ② 荀子 ✗ ③ 孔子 ✓，则相关文档排在第3位 → 分数低

### 1.3.2 Context Recall（上下文召回率）
Ground Truth 中包含的信息，检索回来的文档覆盖了多少？有没有漏掉关键信息？

公式：

$$\text{Context Recall} = \frac{|\text{检索文档能支持的 GT 陈述数}|}{|\text{GT 中总陈述数}|} $$

RAGAS 将 Ground Truth（标准答案） 拆解为原子陈述，逐一判断能否从检索文档中推断出来。

举例：
- 问题："《学记》提出了哪些教学原则？"
- Ground Truth："教学相长、豫时孙摩、长善救失、启发诱导"（4条）
- 检索只返回了"教学相长"和"启发诱导"相关内容 → Context Recall ≈ 2/4 = 0.5（低）

### 1.3.3 Context Entities Recall（事实上下文召回）
Ground Truth 中出现的实体（人名、地名、术语、数字等），检索回来的文档覆盖了多少？适用于事实密集型场景。

公式：

$$\text{Context Entities Recall} = \frac{|\text{RE} \cap \text{RCE}|}{|\text{RE}|} $$

其中 RE = Ground Truth 中的实体集合，RCE = 检索文档中的实体集合。

举例：
- 问题："夸美纽斯有什么贡献？"
- Ground Truth 实体：{夸美纽斯, 大教学论, 班级授课制, 泛智教育, 学年制}
- 检索文档实体：{夸美纽斯, 大教学论, 班级授课制} → 命中3个
- Context Entities Recall = 3/5 = 0.6

### 1.3.4 Faithfulness（忠实度 / 反幻觉）
模型回答中每一个事实陈述是否都能从检索文档中找到依据？检测"幻觉"的核心指标。

公式：

$$\text{Faithfulness} = \frac{|\text{检索文档能支持的陈述数}|}{|\text{回答中的总陈述数}|} $$

RAGAS 将回答拆解为原子级事实陈述，逐一用检索文档验证。

举例：
- 检索文档："夸美纽斯在《大教学论》中提出班级授课制。"
- 模型回答："夸美纽斯提出了班级授课制和泛智教育，被称为教育学之父。"
- 拆解：①"班级授课制" ✓ ②"泛智教育" ✗（文档无） ③"教育学之父" ✗（文档无）
- Faithfulness = 1/3 ≈ 0.33 —— 编造了2个事实

### 1.3.5 Response Relevancy（回答相关性 / 切题度）
模型的回答是否紧扣用户问题？有没有答非所问或跑题？  
| 注：Python 类名为 AnswerRelevancy，文档中称为 Response Relevancy。

公式：

$$\text{Response Relevancy} = \frac{1}{N} \sum_{i=1}^{N} \cos(E_{\text{original\_question}},\  E_{\text{generated\_question}_i}) $$

RAGAS 从模型回答反向生成 N 个问题，计算每个生成问题与原始问题的余弦相似度后取平均。

举例：
- 用户问："教育的本质属性是什么？"
- 回答A："教育是有目的地培养人的社会活动。" → 反向问题 = "教育的本质属性是什么？" → 相似度高
- 回答B："教育经历了原始、古代、现代三个阶段……" → 反向问题 = "教育经历了哪些阶段？" → 相似度低

### 1.3.6 Noise Sensitivity（噪声敏感度）
系统在利用检索文档时，产生错误回答的频率有多高？噪声敏感度越低，系统越鲁棒。

公式：

$$\text{Noise Sensitivity}_{\text{relevant}} = \frac{|\text{回答中的错误陈述数}|}{|\text{回答的总陈述数}|} $$

其中"错误陈述" = 不符合 Ground Truth 或无法归因于相关上下文的陈述。得分越低越好（0 = 完美）。

举例：
- 问题："LIC 以什么著称？"
- Ground Truth："LIC 是印度最大的保险公司，1956年成立，管理大规模投资组合。"
- 模型回答："LIC是最佳保险公司，拥有庞大投资组合，对金融稳定有贡献。"
- 拆解：①"最大保险公司" ✓ ②"投资组合" ✓ ③"对金融稳定有贡献" ✗（Ground Truth 未提及）
- Noise Sensitivity = 1/3 ≈ 0.333

# 2. 准备被评估的RAG系统和数据集
## 2.1 准备知识库和检索组件
### 2.1.1 加载文档

In [1]:
# 加载知识库
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import os

# 1. 加载并切分中二知识笔记
with open("中二知识笔记.md", "r", encoding="utf-8") as f:
    docs = f.read()

# 用md结构切分
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
chunks = markdown_splitter.split_text(docs)

# 创建递归切分器
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 0,
    separators = ["\n\n", "\n", "。"]
)

ds = []
idx = 1

# 定义方法，作用是给文档内容拼上三级标题和id
def handle_doc_header(doc: Document):
    global idx
    m = doc.metadata
    doc.page_content = f"### {m['Header 3']}\n{doc.page_content}"
    doc.id = f"doc_{idx}"
    idx += 1
    return doc

# 遍历文档，判断大小，如果超过200，再用递归大小切分
for doc in chunks:
    m = doc.metadata
    content = doc.page_content
    if len(content) > 200:
        sub_docs = recursive_splitter.split_documents([doc])
        ds.extend(handle_doc_header(d) for d in sub_docs)
    else:
        ds.append(handle_doc_header(doc))

for chunk in ds:
    print(len(chunk.page_content))
    print(chunk.id)
    print(chunk)
    print("=" * 60)

107
doc_1
page_content='### （一）古代希腊教育家
**苏格拉底**：产婆术，通过问答引导学生自己发现真理。
**柏拉图**：主张教育应培养哲学王，强调理性与德性的统一。
**亚里士多德**：提出德智体和谐发展，重视习惯养成与自由教育。' metadata={'Header 1': '第一章 教育概述', 'Header 2': '第一节 中外教育家及其教育思想', 'Header 3': '（一）古代希腊教育家'}
122
doc_2
page_content='### （二）近代西方教育家
**夸美纽斯**：提出班级授课制，著有《大教学论》，主张“把一切事物教给一切人”。
**卢梭**：自然主义教育，强调儿童天性，著有《爱弥儿》。
**杜威**：实用主义教育，主张“从做中学”，教育即生活、教育即生长。' metadata={'Header 1': '第一章 教育概述', 'Header 2': '第一节 中外教育家及其教育思想', 'Header 3': '（二）近代西方教育家'}
191
doc_3
page_content='### （三）孔子及《论语》主要思想
**教育作用**：庶、富、教；性相近，习相远。
**教育对象**：“有教无类”，教育民主思想。
**教育目的**：以完善人格为教育的首要目的，培养士和君子。
**教学内容**：文、行、忠、义。
**教学过程**：学、思、习、行。
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。
**教师观**：其身正，不令而行；其身不正，虽令不从。' metadata={'Header 1': '第一章 教育概述', 'Header 2': '第一节 中外教育家及其教育思想', 'Header 3': '（三）孔子及《论语》主要思想'}
98
doc_4
page_content='### （四）孟子主要思想
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。
**教学原则**：循序渐进，专心有恒。' metadata={'Header 1': '第一章 教育概述', 'Header 2': '第一节 中外教育家及其教育思想', 'Header 3': '（四）孟子主要思想'}
63
doc_5
pag

### 2.1.2 准备知识库和工具

In [2]:
# 2. 向量化
embeddings = DashScopeEmbeddings(
    model = "text-embedding-v4",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents(ds)
retriever = vectorstore.as_retriever(kw_args = {"k": 3})

print(f"知识库已就绪：{len(ds)} 个文档切片")

# 3. BM25 稀疏检索
import bm25s
import jieba

def create_bm25_index(metadata_corpus, index_path="my_index.bm25", k1=1.5, b=0.75):
    if os.path.exists(index_path):
        print(f"索引文件已存在，正在加载：{index_path}")
        _retriever = bm25s.BM25.load(index_path, load_corpus=True)
    else:
        print(f"未找到索引文件，正在创建新索引...")
        # 对语料进行分词
        corpus_tokens = [jieba.lcut(doc["content"]) for doc in metadata_corpus]
        # 基于原始文档创建索引库，将来检索出来的也是原始文档
        _retriever = bm25s.BM25(k1=k1, b=b, corpus=metadata_corpus)
        # 创建索引
        _retriever.index(corpus_tokens)
        # 保存到本地
        _retriever.save(index_path)
        print(f"索引已保存至: {index_path}")

    return _retriever


metadata_corpus = [{"id": doc.id, "content": doc.page_content} for doc in ds]
bm25_retriever = create_bm25_index(metadata_corpus)

def bm25_search(query: str, k: int = 3):
    query_tokens = [jieba.lcut(query)]
    results, scores = bm25_retriever.retrieve(query_tokens, k=k)
    return [(results[0, i], scores[0, i]) for i in range(results.shape[1])]

# 4. RRF 融合
def reciprocal_rank_fusion(ranked_lists, k=60):
    rrf_scores = {}
    results = {}
    for rank_list in ranked_lists:
        for rank, doc in enumerate(rank_list, start=1):
            doc_id = doc["id"]
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (k + rank)
            results[doc_id] = doc
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [results[doc_id] for doc_id, score in sorted_docs]


# 5. Cross-Encoder 重排序
import torch
from sentence_transformers import CrossEncoder

cross_model = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

def cross_encoder_rerank(query: str, docs, top_k: int = 3):
    scores = cross_model.predict([(query, doc["content"]) for doc in docs])
    for i, s in enumerate(scores):
        docs[i]["score"] = float(s)
    sorted_docs = sorted(docs, key=lambda x: x["score"], reverse=True)
    positive_docs = [doc for doc in sorted_docs if doc["score"] > 0]
    return positive_docs[:min(top_k, len(positive_docs))]

Building prefix dict from the default dictionary ...


知识库已就绪：7 个文档切片
未找到索引文件，正在创建新索引...


Dumping model to file cache C:\Users\86180\AppData\Local\Temp\jieba.cache
Loading model cost 0.792 seconds.
Prefix dict has been built successfully.


BM25S Create Vocab:   0%|          | 0/7 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/7 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


## 2.2 创建Agent

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent, AgentState

def retrieve_docs(query: str, top_k: int):
    """混合检索：稠密 + 稀疏 + RRF + CrossEncoder"""
    # 1. 稠密检索
    vector_results = vectorstore.similarity_search(query, k = top_k)
    # 2. 稀疏检索 BM25
    bm25_results_list = bm25_search(query, k = top_k)

    if not vector_results and not bm25_results_list:
        return []

    # 3. RRF融合
    vector_rs = [{"id": doc.id, "content": doc.page_content} for doc in vector_results]
    bm25_rs = [doc for doc, _ in bm25_results_list]
    rrf_results = reciprocal_rank_fusion([vector_rs, bm25_rs])

    # 4. Cross-Encoder 重排序
    return cross_encoder_rerank(query, rrf_results, top_k)


contexts_store = {}

@tool
def search_knowledge_base(query: str, runtime: ToolRuntime[AgentState]) -> str:
    """搜索知识库，获取教育学、名人等的知识。需要查找资料时使用。"""
    final_docs = retrieve_docs(query, 5)
    if not final_docs:
        return "未找到相关文档"

    # 顺手存 context 评估取用
    msg = runtime.state['messages'][0].content
    cs = contexts_store.get(msg, [])
    cs.extend([doc["content"] for doc in final_docs])
    contexts_store[msg] = list(set(cs))

    return "\n\n".join(doc["content"] for doc in final_docs)


rag_agent = create_agent(
    model = "deepseek-v4-flash",
    tools=[search_knowledge_base],
    system_prompt="""
    你是一个教资考试答题助手。你必须严格遵守以下规则：

    1. 有关教资的问题**只能**调用`search_knowledge_base`工具来查阅相关文档，根据文档信息回答问题
    2. **禁止**添加任何你自己的知识、理解或推断
    3. **禁止**扩展、举例或给出建议
    4. 如果参考文档中没有相关信息，直接回复"根据提供的资料，无法回答此问题"
    5. 回答要简洁，直接引用原文，不要改写或总结
    """
)

print("RAG Agent 创建完成！（混合检索：稠密+BM25+RRF+CrossEncoder）")

## 2.3 准备数据集
数据集的每条数据包含4部分：
- user_input：问题
- reference：标准答案
- retrieved_contexts：召回的文档
- response：RAG系统生成的答案

前两个需要提前提供好，后两个则是需要调用RAG系统生成。

In [ ]:
import json
import time
from ragas import Dataset
import os

def create_dataset():
    """创建dataset，先尝试本地加载，如果没有则重新生成"""

    # 1. 创建 ragas Dataset
    dataset = Dataset(name="rag_eval", backend="local/csv", root_dir = "./experiments")
    if os.path.exists("./experiments/datasets/rag_eval.csv"):
        # 尝试读取本地数据
        dataset.reload()
    if len(dataset) > 0:
        print(f"加载了 {len(dataset)} 个测试数据集，不再重新生成")
        return dataset

    # 2. 加载评估问题
    with open("./resources/ragas_eval_dataset.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)

    print(f"加载了 {len(eval_data)} 个评估问题")

    # 3.调用Agent，生成response和context

    # 每次 Agent 调用后的等待时间（秒），避免触发模型提供商限流
    # DeepSeek 免费版建议 1~2s，付费版可适当降低，根据实际报错调整

    REQUEST_DELAY = 1
    print(f"开始收集回答和上下文（共 {len(eval_data)} 题，间隔 {REQUEST_DELAY}s）...")
    print("-" * 50)

    for i, item in enumerate(eval_data):
        query = item["question"]

        response = rag_agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            {"configurable": {"thread_id": f"eval-{i}"}}
        )
        answer = response["messages"][-1].content

        # 从 tool 的 store 中取 context（一次检索，不重复）
        contexts = contexts_store.get(query, [])

        # 写入数据集
        dataset.append({
            "user_input": query,
            "response": answer,
            "retrieved_contexts": contexts,
            "reference": item["ground_truth"]
        })

        # 等待一会儿再发下一个请求，避免触发限流
        if i < len(eval_data) - 1:  # 最后一条不用等
            time.sleep(REQUEST_DELAY)

    dataset.save()
    print(f"\n评估数据集已保存: {len(eval_data)} 个样本 -> ./experiments/rag_eval/")
    return dataset

dataset = create_dataset()

# 3. RAGAS 评估
RAGAS 需要 LLM 和 Embedding 模型作为评判器（Judge），推荐用强模型以确保评估可靠性。最新版本中默认是基于OpenAI的客户端协议，虽然可配置但也仅支持国外的几种模型协议。所以我们需要手动初始化。

## 3.1 准备模型
由于RAGAS默认支持OpenAI的客户端协议，所以我们不能用LangChain来初始化。而是改用OpenAI的客户端。

In [ ]:
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from openai import AsyncOpenAI

llm_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

embedding_client = AsyncOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

# 文本推理模型
evaluator_llm = llm_factory(
    model="deepseek-chat",
    client=llm_client,
    max_tokens = 384000
)

# 向量模型
evaluator_embeddings = embedding_factory(
    model = "text-embedding-v3",
    client=embedding_client
)

## 3.2 指标评估

In [ ]:
row = dataset.__getitem__(3)
user_input = row.get("user_input")
response = row.get("response")
retrieved_contexts = row.get("retrieved_contexts")
reference = row.get("reference")

In [ ]:
# 6个核心指标（从 collections 导入）
from ragas.metrics.collections import (
    Faithfulness,  # 忠实度
    AnswerRelevancy,  # 回答相关性
    ContextPrecision,  # 上下文精度
    ContextEntityRecall,  # 上下文实体召回
    NoiseSensitivity,  # 噪声敏感度
    ContextRecall  # 上下文召回
)

print(f"========开始评估：input:{user_input}===========")
# ---- 检索阶段指标 ----
cp = ContextPrecision(llm=evaluator_llm)

cp_result = await cp.ascore(
    user_input=user_input,
    reference=reference,
    retrieved_contexts=retrieved_contexts,
)

cr = ContextRecall(llm=evaluator_llm)
cr_result = await cr.ascore(
    user_input=user_input,
    reference=reference,
    retrieved_contexts=retrieved_contexts,
)

cer = ContextEntityRecall(llm=evaluator_llm)
cer_result = await cer.ascore(
    reference=reference,
    retrieved_contexts=retrieved_contexts,
)

# ---- 生成阶段指标 ----
faith = Faithfulness(llm=evaluator_llm)

faith_result = await faith.ascore(
    user_input=user_input,
    response=response,
    retrieved_contexts=retrieved_contexts,
)

ar = AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
ar_result = await ar.ascore(
    user_input=user_input,
    response=response,
)

ns = NoiseSensitivity(llm=evaluator_llm)
ns_result = await ns.ascore(
    user_input=user_input,
    response=response,
    reference=reference,
    retrieved_contexts=retrieved_contexts,
)

## 3.3 定义experiment
新版的RAGAS中，把评估任务称为experiment，也就是实验。之所以叫这个名字，是因为所有的实验数据都是流痕的，可追溯的，不像之前老版本中是黑盒打分。

创建实验其实就是定义函数，但要用$@experiment$装饰器。

In [ ]:
# 新 API：@experiment 装饰器 + 标准指标
from ragas import experiment
from pydantic import BaseModel

# 定义实验结果模型
class EvalResult(BaseModel):
    context_recall: float
    Faithfulness: float

# 定义实验：逐行评分
@experiment(experiment_model=EvalResult)
async def run_rag_eval(row):
    # 先把评估的数据准备好
    user_input = row.get("user_input")
    reference = row.get("reference")
    retrieved_contexts = row.get("retrieved_contexts")
    response = row.get("response")

    # ---- 检索阶段指标 ----
    cr = ContextRecall(llm=evaluator_llm)
    cr_result = await cr.ascore(
        user_input=user_input,
        reference=reference,
        retrieved_contexts=retrieved_contexts
    )

    faith = Faithfulness(llm=evaluator_llm)
    faith_result = await faith.ascore(
        user_input=user_input,
        response=response,
        retrieved_contexts=retrieved_contexts
    )

    return EvalResult(
        context_recall=cr_result.value,
        Faithfulness=faith_result.value
    )

3.4 批量评估

In [ ]:
train_dataset, test_dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
# 把数据集的评估name设定与原来一致，这样就会保存到dataset一样的目录，方便查看
tds = Dataset.from_pandas(test_dataset.to_pandas(), name="rag_eval", backend="local/csv", root_dir="./experiments")

# 开始评估
results = await run_rag_eval.arun(tds, name="rag_baseline")

## 3.5 评估结果解读

| 指标 | 发现问题 | 优化方向 |
|:---|:---|:---|
| Faithfulness 低 | 模型在编造，没有基于文档回答 | 加强系统提示、降低 temperature |
| Context Precision 低 | 检索返回了无关内容（噪声多） | 优化 chunk_size、加入 Query Rewriting |
| Context Recall 低 | 需要的信息没有检索到（缺信息） | 调整 embedding 模型、增大 k、加入 Hybrid Search |
| Context Entities Recall 低 | 关键实体（人名、术语、数字）未覆盖 | 加入 BM25 关键词检索、优化 chunk 策略 |
| Answer Relevancy 低 | 模型回答跑题了 | 优化 Prompt、限制回答范围 |
| Noise Sensitivity 高 | 噪声文档导致回答出错 | 加入 Cross-Encoder 重排序、提升检索精度 |